# Meta-OpenENV Real Unsloth RL Pipeline

This notebook executes a true Proximal Policy Optimization (PPO) training loop against the live Meta-OpenENV FastAPI backend. It fine-tunes the **Qwen 2.5 0.5B** model to defend against adversarial tickets.

In [1]:
!pip install unsloth trl peft torch requests matplotlib numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [2]:
import os
import json
import requests
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

%matplotlib inline
plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True})


/tmp/ipykernel_2231/315658682.py:8: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
API_BASE = "https://subhakshay-phase2.hf.space"

def reset_env(task_id=2, difficulty_init=1):
    res = requests.post(f"{API_BASE}/reset", json={
        "task_id": task_id,
        "difficulty_init": difficulty_init,
        "attacker_enabled": True
    })
    return res.json()

def step_env(session_id, action):
    res = requests.post(f"{API_BASE}/step", json={
        "session_id": session_id,
        "action": action
    })
    return res.json()

### 2. Load Model via Unsloth & Initialize TRL

In [4]:
max_seq_length = 1024
dtype = None
load_in_4bit = True

print("Loading Qwen-2.5-0.5B via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)



optimizer = torch.optim.AdamW(model.parameters(), lr=1.41e-5)


Loading Qwen-2.5-0.5B via Unsloth...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


### 3. Prompting & Action Parsing

In [5]:
def format_prompt(obs):
    subject = obs.get("subject", "")
    body = obs.get("body", "")
    tier = obs.get("tier", "Bronze")
    return f"""<|im_start|>system
You are an AI customer support agent. Determine the priority (Low, Medium, High, Critical) and category (Technical, Billing, Account). Format your output strictly as JSON.
<|im_end|>
<|im_start|>user
Tier: {tier}
Subject: {subject}
Body: {body}
<|im_end|>
<|im_start|>assistant
```json
"""

def parse_action(text):
    try:
        json_str = text.split("```json")[1].split("```")[0].strip()
        return json.loads(json_str)
    except:
        # Fallback action if model fails to output valid JSON
        return {
            "assign_priority": "Medium",
            "assign_category": "Technical",
            "draft_response": "Thank you for reaching out. We will look into this.",
            "escalate": False
        }


### 4. Actual Training Loop

In [13]:
NUM_EPISODES = 40
MAX_STEPS = 30

# Tracking metrics
metrics = {
    "episodes": [],
    "mean_rewards": [],
    "attacker_win_rates": [],
    "balances": [],
    "sla_breaches": [],
    "difficulty": [],
}

for episode in range(1, NUM_EPISODES + 1):
    print(f"\n=== Starting Episode {episode} ===")
    session = reset_env()
    sid = session.get("session_id")
    if not sid:
        print("Failed to connect to API Backend. Is it running?")
        break

    obs = session["observation"]

    episode_rewards = []
    attacker_wins = 0
    done = False
    step = 0

    while not done and step < MAX_STEPS:
        prompt = format_prompt(obs)
        query_tensor = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

        # Generation
        generation_kwargs = {
            "min_length": -1,
            "top_k": 0.0,
            "top_p": 1.0,
            "do_sample": True,
            "pad_token_id": tokenizer.eos_token_id,
            "max_new_tokens": 128
        }
        # Generate the response tensor (contains input_ids for the full generated sequence)
        response_tensor = model.generate(query_tensor, **generation_kwargs)

        # Clone and detach the response_tensor to ensure it's not an 'inference tensor'
        # for subsequent operations that require gradients. This breaks its association
        # with the inference context where it was created.
        response_tensor_for_loss = response_tensor.clone().detach()

        # Decode only the newly generated tokens for action parsing
        new_tokens_ids = response_tensor[0][query_tensor.shape[1]:]
        response_text = tokenizer.decode(new_tokens_ids, skip_special_tokens=True)

        action = parse_action(response_text)

        # Step Environment
        result = step_env(sid, action)
        reward = result.get("reward", 0.0)
        done = result.get("done", True)
        obs = result.get("observation", {})
        ws = result.get("world_state", {})

        episode_rewards.append(reward)
        if reward < 0: attacker_wins += 1

        # Manual REINFORCE Update step
        model.train() # Ensure model is in training mode
        optimizer.zero_grad()

        # Forward pass on the generated sequence to get log probs.
        # This forward pass must track gradients for model parameters.
        # Use the cloned tensor as input to avoid the 'inference tensor' error.
        outputs = model(response_tensor_for_loss)
        logits = outputs.logits

        # We only care about the log probs of the newly generated tokens
        prompt_length = query_tensor.shape[1]
        shift_logits = logits[0, prompt_length-1:-1, :].contiguous()
        # Use the cloned tensor for labels as well, ensures consistency and avoids inference tensor issues.
        shift_labels = response_tensor_for_loss[0, prompt_length:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        # CrossEntropyLoss returns -log_prob. We want to minimize -log_prob * reward
        # which is equivalent to maximizing log_prob * reward.
        token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        pg_loss = (token_losses * reward).mean()

        pg_loss.backward()
        optimizer.step()
        model.eval()

        step += 1
        print(f"  Step {step} | Reward: {reward:.3f} | Balance: ${ws.get('company_balance', 0)}")

    # Record Episode Metrics
    mean_rew = np.mean(episode_rewards) if episode_rewards else 0
    win_rate = attacker_wins / step if step > 0 else 0
    metrics["episodes"].append(episode)
    metrics["mean_rewards"].append(mean_rew)
    metrics["attacker_win_rates"].append(win_rate)
    metrics["balances"].append(ws.get("company_balance", 0))
    metrics["sla_breaches"].append(ws.get("sla_breaches", 0))
    metrics["difficulty"].append(ws.get("difficulty_level", 1))

    print(f"> Episode Complete | Mean Reward: {mean_rew:.3f} | Attacker WR: {win_rate:.1%}")


=== Starting Episode 1 ===


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

  Step 1 | Reward: 2.790 | Balance: $10000.0


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 2 | Reward: 4.290 | Balance: $10000.0
  Step 3 | Reward: -6.840 | Balance: $10000.0
> Episode Complete | Mean Reward: 0.080 | Attacker WR: 33.3%

=== Starting Episode 2 ===


...

  Step 3 | Reward: 5.660 | Balance: $10000.0
  Step 4 | Reward: -5.040 | Balance: $10000.0
> Episode Complete | Mean Reward: 1.550 | Attacker WR: 25.0%

=== Starting Episode 40 ===


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 1 | Reward: 2.790 | Balance: $10000.0


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 2 | Reward: 2.790 | Balance: $10000.0
  Step 3 | Reward: -6.840 | Balance: $10000.0
> Episode Complete | Mean Reward: -0.420 | Attacker WR: 33.3%
